### Overview

This notebook builds conversation-level features and target variables from the English-only ShareGPT subset created in Notebook 1.

The goal is to transform tree-structured dialogue data into a structured analytical dataset for downstream modeling.  
Each conversation is mapped to exactly one row.

Pipeline Summary

The workflow consists of the following stages:

- Data Loading  
The filtered English-only dataset is loaded from the processed data directory.

- Feature Extraction  
Conversation-level features are created from the first prompt-response pair, dialogue structure, token usage, prompt design, task cues, and orthographic quality.

- Prompt Validation  
Broken, empty, or noisy prompts are removed before representation learning.

- Embedding  
Prompt embeddings are generated using a sentence-transformer model.

- Topic Modelling  
Semantic prompt topics are inferred using BERTopic.

- Target Construction  
Proxy target variables are derived for downstream efficiency and quality modeling.

- Export  
The full dataframe, feature matrix, target table, and embeddings are saved for later notebooks.


### Design Rationale

This notebook prioritizes:

- Clear separation of feature logic and target logic
- Reproducibility through deterministic preprocessing steps
- Modular helper functions
- Consistency with Notebook 1
- Readability for later model interpretation

In [2]:
import json
import re
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import tiktoken
from spellchecker import SpellChecker
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_distances
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

pd.set_option("display.max_colwidth", None)

# ----------------------------
# CONFIG
# ----------------------------


PROJECT_ROOT = Path.cwd().parent

CONFIG = {
    "processed_dir": PROJECT_ROOT / "01_data" / "02_processed",
    "features_dir": PROJECT_ROOT / "01_data" / "03_features",
    "input_file": "sharegpt_english.json",
    "full_dataframe_file": "conversation_dataframe.csv",
    "features_file": "conversation_features.csv",
    "target_file": "conversation_target.csv",
    "embeddings_file": "embeddings.npy",
    "embedding_model": "all-MiniLM-L6-v2",
    "embedding_chunk_size": 5000,
    "embedding_batch_size": 64,
    "min_prompt_length": 10,
    "random_state": 42,
}

INPUT_PATH = CONFIG["processed_dir"] / CONFIG["input_file"]
FULL_DF_PATH = CONFIG["features_dir"] / CONFIG["full_dataframe_file"]
FEATURES_PATH = CONFIG["features_dir"] / CONFIG["features_file"]
TARGET_PATH = CONFIG["features_dir"] / CONFIG["target_file"]
EMBEDDINGS_PATH = CONFIG["features_dir"] / CONFIG["embeddings_file"]

CONFIG["features_dir"].mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_PATH:", INPUT_PATH)
print("Exists:", INPUT_PATH.exists())


# ----------------------------
# GLOBAL OBJECTS
# ----------------------------

WORD_RE = re.compile(r"[A-Za-z']+")
ENCODING = tiktoken.get_encoding("cl100k_base")
SPELL = SpellChecker(language="en")

c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis
INPUT_PATH: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\02_processed\sharegpt_english.json
Exists: True


In [3]:
 

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

In [4]:
# ----------------------------
# CORE EXTRACTION
# ----------------------------

def get_first_user_prompt(tree):
    conversations = tree.get("conversations", [])

    for message in conversations:
        if message.get("from") == "human":
            return message.get("value", "")

    return ""


def get_first_prompt_response_pair(tree):
    conversations = tree.get("conversations", [])

    for i in range(len(conversations) - 1):
        current = conversations[i]
        next_msg = conversations[i + 1]

        if current.get("from") == "human" and next_msg.get("from") == "gpt":
            return current.get("value", ""), next_msg.get("value", "")

    return "", ""


def get_words(text):
    return WORD_RE.findall(str(text).lower())

In [5]:
# ----------------------------
# TOKEN AND STRUCTURE FEATURES
# ----------------------------

def count_tokens(text):
    return len(
        ENCODING.encode(
            str(text),
            disallowed_special=()
        )
    )


def count_user_prompts(tree):
    return sum(
        message.get("from") == "human"
        for message in tree.get("conversations", [])
    )


def count_total_user_tokens(tree):
    total = 0

    for message in tree.get("conversations", []):
        if message.get("from") == "human":
            total += count_tokens(message.get("value", ""))

    return total


def count_total_assistant_tokens(tree):
    total = 0

    for message in tree.get("conversations", []):
        if message.get("from") == "gpt":
            total += count_tokens(message.get("value", ""))

    return total

In [6]:
# ----------------------------
# PROMPT DESIGN FEATURES
# ----------------------------

def has_role_instruction(prompt):
    prompt = str(prompt).lower()

    role_patterns = [
        "act as",
        "you are",
        "you're",
        "pretend you are",
        "imagine you are",
        "take the role",
        "role of",
    ]

    return any(pattern in prompt for pattern in role_patterns)


def has_audience_or_level_instruction(prompt):
    prompt = str(prompt).lower()

    patterns = [
        "explain it to me like",
        "explain like i am",
        "explain like i'm",
        "eli5",
        "like i am 5",
        "like i'm 5",
        "like i am 10",
        "like i'm 10",
        "for beginners",
        "to a beginner",
        "in simple terms",
        "plain english",
        "non-technical",
        "for a child",
        "for a high school student",
    ]

    return any(pattern in prompt for pattern in patterns)


def has_format_instruction(prompt):
    prompt = str(prompt).lower()

    format_keywords = [
        "bullet point",
        "bullet points",
        "table",
        "json",
        "csv",
        "list",
        "markdown",
        "format",
        "section",
        "sections",
        "outline",
        "step by step",
        "numbered",
    ]

    return any(keyword in prompt for keyword in format_keywords)


def count_questions(prompt):
    return str(prompt).count("?")


def prompt_style(prompt):
    prompt_lower = str(prompt).lower().strip()

    instruction_verbs = [
        "write", "summarize", "explain", "create", "generate", "translate",
        "list", "compare", "analyze", "give", "make", "draft", "compose",
        "rewrite", "classify", "extract", "find", "calculate",
    ]

    if "?" in prompt_lower:
        return "question"

    if any(prompt_lower.startswith(verb) for verb in instruction_verbs):
        return "instruction"

    return "other"

In [7]:
# ----------------------------
# TEXT QUALITY AND TASK TYPE
# ----------------------------

def orthographic_error_rate(prompt):
    words = get_words(prompt)

    if not words:
        return 0.0

    unknown_words = SPELL.unknown(words)

    return len(unknown_words) / len(words)


def classify_task_type(prompt):
    prompt = str(prompt).lower()

    if any(word in prompt for word in [
        "python", "java", "javascript", "typescript", "sql", "html", "css",
        "api", "debug", "error", "exception", "bash", "terminal",
        "command", "linux", "c++", "cpp"
    ]):
        return "coding"

    if any(word in prompt for word in [
        "email", "subject line", "reply to", "newsletter"
    ]):
        return "email_writing"

    if any(word in prompt for word in [
        "summarize", "summary", "tl;dr", "key points"
    ]):
        return "summarization"

    if any(word in prompt for word in [
        "translate", "translation", "in english", "into english",
        "into spanish", "into french"
    ]):
        return "translation"

    if any(word in prompt for word in [
        "explain", "what is", "how does", "why does",
        "difference between", "simple terms"
    ]):
        return "explanation"

    if any(word in prompt for word in [
        "write", "draft", "compose", "story", "poem",
        "article", "script", "blog post"
    ]):
        return "writing_generation"

    if any(word in prompt for word in [
        "idea", "ideas", "brainstorm", "suggest", "recommend"
    ]):
        return "brainstorming"

    if any(word in prompt for word in [
        "act as", "pretend", "roleplay", "you are now", "simulate"
    ]):
        return "roleplay"

    return "general_assistance"

In [8]:
# ----------------------------
# PROMPT VALIDATION
# ----------------------------

def is_valid_prompt(text):
    text = str(text)

    if not text or len(text) < CONFIG["min_prompt_length"]:
        return False

    noise_markers = [
        "root@",
        "-----",
        "Enter pass phrase",
        "Select an option",
    ]

    return not any(marker in text for marker in noise_markers)

In [9]:
# ----------------------------
# CONVERSATION TO ROW
# ----------------------------

def build_conversation_row(tree):
    first_prompt, first_response = get_first_prompt_response_pair(tree)

    total_turns = len(tree.get("conversations", []))
    total_user_prompts = count_user_prompts(tree)

    total_user_tokens = count_total_user_tokens(tree)
    total_assistant_tokens = count_total_assistant_tokens(tree)
    total_tokens = total_user_tokens + total_assistant_tokens

    follow_up_prompts = max(0, total_user_prompts - 1)
    needs_follow_up = int(follow_up_prompts > 0)

    return {
        "conversation_id": tree.get("id"),

        "first_prompt": first_prompt,
        "first_response": first_response,

        "first_prompt_tokens": count_tokens(first_prompt),
        "first_response_tokens": count_tokens(first_response),

        "total_turns": total_turns,
        "interaction_rounds": total_turns / 2,
        "total_user_prompts": total_user_prompts,
        "follow_up_prompts": follow_up_prompts,
        "needs_follow_up": needs_follow_up,

        "total_user_tokens": total_user_tokens,
        "total_assistant_tokens": total_assistant_tokens,
        "total_tokens": total_tokens,
        "log_total_tokens": np.log1p(total_tokens),

        "has_role_instruction": int(has_role_instruction(first_prompt)),
        "has_audience_or_level_instruction": int(has_audience_or_level_instruction(first_prompt)),
        "has_format_instruction": int(has_format_instruction(first_prompt)),
        "question_count": int(count_questions(first_prompt)),
        "prompt_style": prompt_style(first_prompt),

        "task_type": classify_task_type(first_prompt),
        "orthographic_error_rate": orthographic_error_rate(first_prompt),
    }

In [10]:
# ----------------------------
# BUILD DATAFRAME
# ----------------------------

def build_dataframe(data):
    rows = [build_conversation_row(tree) for tree in data]
    return pd.DataFrame(rows)

In [11]:
# ----------------------------
# EMBEDDING
# ----------------------------

def embed_chunked(texts, model, chunk_size=5000, batch_size=64):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), chunk_size)):
        chunk = texts[i:i + chunk_size]

        emb = model.encode(
            chunk,
            batch_size=batch_size,
            show_progress_bar=False
        )

        all_embeddings.append(emb)

    return np.concatenate(all_embeddings, axis=0)

In [12]:
# ----------------------------
# LOAD DATA
# ----------------------------

english_data = load_json(INPUT_PATH)

print(f"Loaded conversations: {len(english_data)}")

# ----------------------------
# CHECK CONVERSATION IDS
# ----------------------------

conversation_ids = [
    item.get("id")
    for item in english_data
    if isinstance(item, dict) and item.get("id") is not None
]

print(f"Conversation IDs: {len(conversation_ids)}")
print(f"Unique Conversation IDs: {len(set(conversation_ids))}")
print(f"Duplicates: {len(conversation_ids) - len(set(conversation_ids))}")

# ----------------------------
# BUILD FEATURES
# ----------------------------

start = perf_counter()

df = build_dataframe(english_data)

duration = perf_counter() - start

print(f"Rows built: {len(df)}")
print(f"Time: {duration:.2f}s")
print(f"Rows/sec: {len(df)/duration:.0f}")

Loaded conversations: 56024
Conversation IDs: 56024
Unique Conversation IDs: 56024
Duplicates: 0
Rows built: 56024
Time: 397.09s
Rows/sec: 141


In [13]:
# ----------------------------
# FILTER INVALID PROMPTS
# ----------------------------

before = len(df)

df["is_valid_prompt"] = df["first_prompt"].apply(is_valid_prompt).astype(int)

df = df[
    df["first_prompt"].fillna("").str.len().gt(0) &
    df["first_response"].fillna("").str.len().gt(0) &
    df["is_valid_prompt"].eq(1)
].reset_index(drop=True)

duration = perf_counter() - start

print(f"Before filter: {before}")
print(f"After filter: {len(df)}")
print(f"Removed: {before - len(df)}")
print(f"Time: {duration:.2f}s")


Before filter: 56024
After filter: 54444
Removed: 1580
Time: 397.64s


In [14]:
# ----------------------------
# RUN EMBEDDING
# ----------------------------

embedder = SentenceTransformer(CONFIG["embedding_model"])

prompts = df["first_prompt"].fillna("").astype(str).to_numpy()

X_emb = embed_chunked(
    prompts,
    model=embedder,
    chunk_size=CONFIG["embedding_chunk_size"],
    batch_size=CONFIG["embedding_batch_size"]
)

assert len(X_emb) == len(df)

np.save(EMBEDDINGS_PATH, X_emb)

print("Embeddings shape:", X_emb.shape)

100%|██████████| 11/11 [24:26<00:00, 133.36s/it]

Embeddings shape: (54444, 384)


In [15]:
# ----------------------------
# TOPIC MODELLING
# ----------------------------

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=CONFIG["random_state"]
)

hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedder,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    min_topic_size=50,
    nr_topics="auto"
)

topics, probs = topic_model.fit_transform(df["first_prompt"].tolist())

df["topic"] = topics
df["topic_prob"] = probs.max(axis=1)

In [16]:
# df["topic"].value_counts(normalize=True)
df["topic"].value_counts()

topic
-1      27958
 0       6786
 1       2197
 2       1483
 3        943
        ...  
 102       54
 101       54
 103       53
 104       53
 105       52
Name: count, Length: 107, dtype: int64

In [17]:
def inspect_topic(topic_model, topic_id, docs, top_n=10, sample_n=5):
    print(f"Topic {topic_id}")
    print("-" * 80)

    words = topic_model.get_topic(topic_id)
    if words:
        print("Top words:")
        for word, weight in words[:top_n]:
            print(f"  {word:20s} {weight:.4f}")
    else:
        print("No words found for this topic.")

    print("\nExample documents:")
    topic_docs = [doc for doc, topic in zip(docs, topic_model.topics_) if topic == topic_id]
    for i, doc in enumerate(topic_docs[:sample_n], start=1):
        print(f"\n[{i}] {doc[:500]}")

In [28]:
topic_labels = topic_model.generate_topic_labels(nr_words=5)

label_map = {
    topic_id: label
    for topic_id, label in zip(sorted(topic_model.get_topics().keys()), topic_labels)
}

df["topic_label"] = df["topic"].map(label_map)

df[["topic", "topic_label"]].head(20)

,topic,topic_label
0,0,0_dan_chatgpt_mode_prompt_developer
1,0,0_dan_chatgpt_mode_prompt_developer
2,-1,-1_like_data_use_write_time
3,23,23_word_words_letters_guess_letter
4,0,0_dan_chatgpt_mode_prompt_developer
5,3,3_singleton_python_int_return_code
6,-1,-1_like_data_use_write_time
7,6,6_gt_null_select_table_gt gt
8,-1,-1_like_data_use_write_time
9,-1,-1_like_data_use_write_time


In [19]:
# ----------------------------
# EMBEDDING NOVELTY
# ----------------------------

centroid = X_emb.mean(axis=0)

df["embedding_novelty"] = cosine_distances(
    X_emb,
    centroid.reshape(1, -1)
).flatten()

In [20]:
# ----------------------------
# TARGETS
# ----------------------------

df["target_cost"] = df["first_response_tokens"]

refusal_pattern = r"(?i)(i can't|i cannot|i’m sorry|i am sorry|as an ai|i do not have access)"
df["target_success"] = (
    ~df["first_response"].fillna("").str.contains(refusal_pattern, regex=True)
).astype(int)

In [30]:
print(df.columns.tolist())

['conversation_id', 'first_prompt', 'first_response', 'first_prompt_tokens', 'first_response_tokens', 'total_turns', 'interaction_rounds', 'total_user_prompts', 'follow_up_prompts', 'needs_follow_up', 'total_user_tokens', 'total_assistant_tokens', 'total_tokens', 'log_total_tokens', 'has_role_instruction', 'has_audience_or_level_instruction', 'has_format_instruction', 'question_count', 'prompt_style', 'task_type', 'orthographic_error_rate', 'is_valid_prompt', 'topic', 'topic_prob', 'embedding_novelty', 'target_cost', 'target_success', 'topic_label']


In [31]:
# ----------------------------
# FEATURE / TARGET SPLIT
# ----------------------------

FEATURE_COLUMNS = [
    "conversation_id",
    "first_prompt",
    "first_response",
    "first_prompt_tokens",
    "first_response_tokens",
    "total_turns",
    "interaction_rounds",
    "total_user_prompts",
    "follow_up_prompts",
    "needs_follow_up",
    "total_user_tokens",
    "total_assistant_tokens",
    "total_tokens",
    "log_total_tokens",
    "has_role_instruction",
    "has_audience_or_level_instruction",
    "has_format_instruction",
    "question_count",
    "prompt_style",
    "task_type",
    "orthographic_error_rate",
    "is_valid_prompt",
    "topic",
    "topic_prob",
    "topic_label",
    "embedding_novelty",
]

TARGET_COLUMNS = [
    "conversation_id",
    "target_cost",
    "target_success",
]

df_features = df[FEATURE_COLUMNS].copy()
df_target = df[TARGET_COLUMNS].copy()

In [32]:
# ----------------------------
# SAVE
# ----------------------------

save_csv(df, FULL_DF_PATH)
save_csv(df_features, FEATURES_PATH)
save_csv(df_target, TARGET_PATH)

print(f"Saved full dataframe: {FULL_DF_PATH}")
print(f"Saved features: {FEATURES_PATH}")
print(f"Saved targets: {TARGET_PATH}")
print(f"Saved embeddings: {EMBEDDINGS_PATH}")

Saved full dataframe: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\03_features\conversation_dataframe.csv
Saved features: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\03_features\conversation_features.csv
Saved targets: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\03_features\conversation_target.csv
Saved embeddings: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\03_features\embeddings.npy


In [33]:
# ----------------------------
# DIAGNOSTICS
# ----------------------------

print("Full dataframe shape:", df.shape)
print("Feature dataframe shape:", df_features.shape)
print("Target dataframe shape:", df_target.shape)
print()
print(df_target["target_success"].value_counts(dropna=False))
print()
print(df_target["target_cost"].describe())
print()
print(df[["first_prompt", "task_type", "prompt_style", "topic_label", "target_success"]].head(10))

Full dataframe shape: (54444, 28)
Feature dataframe shape: (54444, 26)
Target dataframe shape: (54444, 3)

target_success
1    52039
0     2405
Name: count, dtype: int64

count    54444.000000
mean       494.628334
std        919.482995
min          1.000000
25%        155.000000
50%        329.000000
75%        580.000000
max      76828.000000
Name: target_cost, dtype: float64

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          